# Theta plots

Load a saved theta run, preview one unit and export PNG/SVG figures for all saved units. The LFP reference plot uses its saved spectrum and excerpt. No raw LFP, tracking, CB/CD results or analysis calculations are needed. Run on `hhw9l84` with the `rfmapping` kernel.


In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from Utils.theta_io import load_theta_analysis
from Utils.theta_plotting import plot_theta_unit, plot_lfp_reference
from Utils.plotting import apply_light_plot_style

apply_light_plot_style()


In [ ]:
recording_root = Path("/mnt/senzailab/Kai/#Recording/m20")
date = "260918"
recording_number = 11
probe_name = "A"
phase_key = "baseline"


session_id = f"{recording_root.name}_{date}_{recording_number}"
session_dir = recording_root / date / f"{date}_{recording_number}"
lfp_channel = 0
result_dir = session_dir / "data" / "theta" / f"Probe{probe_name}" / phase_key / f"channel{lfp_channel}"
plot_directory = result_dir / "figures"
unit_id = 194
is_save = True
theta_results = load_theta_analysis(result_dir)
display(theta_results["summary"])


## Saved LFP reference


In [ ]:
if theta_results["lfp_reference"] is None:
    print("This run has no saved LFP reference preview.")
else:
    figure = plot_lfp_reference(
        {session_id: theta_results["lfp_reference"]},
        channel=theta_results["metadata"].get("reference_channel", "unspecified"),
    )
    if is_save:
        plot_directory.mkdir(parents=True, exist_ok=True)
        for suffix in ("png", "svg"):
            figure.savefig(plot_directory / f"lfp_reference.{suffix}", dpi=200, facecolor="white", transparent=False)
    display(figure)
    plt.close(figure)


## Export all saved units and preview one

The summary table retains nullable classifications and the saved BH q values. Units without enough spikes have empty diagnostic panels labeled unassessed. A selected unit must belong to this saved run.


In [ ]:
if is_save:
    unit_directory = plot_directory / "all_units"
    unit_directory.mkdir(parents=True, exist_ok=True)
    for saved_unit in theta_results["summary"].index:
        figure = plot_theta_unit(
            theta_results["units"].get(saved_unit, {}),
            title=f"{session_id} | Unit {saved_unit}",
            summary=theta_results["summary"].loc[saved_unit],
        )
        for suffix in ("png", "svg"):
            figure.savefig(unit_directory / f"unit_{saved_unit}.{suffix}", dpi=200, facecolor="white", transparent=False)
        plt.close(figure)
    print(f"Saved unit figures: {unit_directory}")

if theta_results["summary"].empty:
    print("This run has no saved units.")
else:
    display(theta_results["summary"].loc[[unit_id]])
    figure = plot_theta_unit(
        theta_results["units"].get(unit_id, {}), title=f"{session_id} | Unit {unit_id}",
        summary=theta_results["summary"].loc[unit_id],
    )
    display(figure)
    plt.close(figure)
